# Pruebas Predicción — Gráficos exportables a PDF

Genera el mismo gráfico de ventana histórica que Streamlit, pero en matplotlib estático para exportar a PDF.

**Capas del gráfico:**
- Puntos: adquisiciones reales S2 (crudo)
- Línea sólida: serie suavizada (Whittaker), verde EVI / azul LSWI
- Línea punteada: predicción/proyección, mismo color
- Línea gris: serie suavizada observada como referencia (validación)
- Línea vertical roja: fecha límite de la ventana de predicción

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter, MonthLocator

from pipeline.ingesta import cargar_indices_desde_bd
from pipeline.modulo_vpm import preprocesar_indices_vpm
from utils.conexionDB import set_db_path, get_db_path, get_connection_raw
from config import GPKG_PATH, GPKG_PRUEBAS_PATH

In [ ]:
# ── 2. Conexión a BD ─────────────────────────────────────────────────────────
set_db_path(GPKG_PRUEBAS_PATH)
print(f"BD activa: {get_db_path()}")

---
### Función de gráfico estático

Replica la misma estructura visual que `_figura_series` de `components/graficas_series.py`, pero con matplotlib para exportación a PDF.

In [ ]:
# ── 3. Función: graficar_prediccion_estatica ───────────────────────────────────
COLORES = {"EVI": "#2ecc71", "LSWI": "#3498db"}


def graficar_prediccion_estatica(
    datos: dict,
    indices: list[str] | None = None,
    extrapolado: dict | None = None,
    validacion: dict | None = None,
    ventana_fecha: pd.Timestamp | None = None,
    ventana_nombre: str | None = None,
    sos_fecha: pd.Timestamp | None = None,
    eos_fecha: pd.Timestamp | None = None,
    titulo: str = "Series temporales",
    figsize: tuple = (14, 8),
) -> tuple[plt.Figure, list[plt.Axes]]:
    """
    Genera un gráfico matplotlib estático con la misma estructura visual que
    la función Streamlit `_figura_series`.

    Parámetros
    ----------
    datos : dict
        ``{"raw": {"EVI": Series, ...}, "smoothed": {"EVI": Series, ...}}``
    indices : list[str], opcional
        Índices a graficar (default ["EVI", "LSWI"]).
    extrapolado : dict, opcional
        ``{"EVI": Series, "LSWI": Series}`` con la proyección.
    validacion : dict, opcional
        ``{"EVI": Series, "LSWI": Series}`` con lo observado post-ventana (gris).
    ventana_fecha : pd.Timestamp, opcional
        Fecha límite de la ventana de predicción (línea roja vertical).
    ventana_nombre : str, opcional
        Nombre de la ventana (etiqueta en la línea vertical).
    sos_fecha, eos_fecha : pd.Timestamp, opcional
        Líneas verticales de inicio/fin de ciclo.
    titulo : str
        Título del gráfico.
    figsize : tuple
        Tamaño de la figura.

    Retorna
    -------
    tuple[plt.Figure, list[plt.Axes]]
        Figura y lista de ejes (uno por índice).
    """
    if indices is None:
        indices = ["EVI", "LSWI"]

    n = len(indices)
    fig, axes = plt.subplots(n, 1, figsize=figsize, sharex=True, squeeze=False)
    axes = axes[:, 0]

    date_fmt = DateFormatter("%b\n%Y")
    locator = MonthLocator()

    for i, idx in enumerate(indices):
        ax = axes[i]
        color = COLORES.get(idx, "#95a5a6")

        raw = datos.get("raw", {}).get(idx)
        smooth = datos.get("smoothed", {}).get(idx)

        # (A) Puntos crudos
        if raw is not None and not raw.empty:
            ax.scatter(raw.index, raw.values,
                       color=color, s=18, alpha=0.5, zorder=3,
                       label=f"{idx} crudo")

        # (B) Serie suavizada (sólida)
        if smooth is not None and not smooth.empty:
            ax.plot(smooth.index, smooth.values,
                    color=color, linewidth=2, zorder=4,
                    label=f"{idx} suavizado")

        # (C) Validación (gris)
        if validacion is not None:
            val = validacion.get(idx)
            if val is not None and not val.empty:
                ax.plot(val.index, val.values,
                        color="#888888", linewidth=2, zorder=3,
                        label=f"{idx} observado (real)")

        # (D) Extrapolado / predicción (punteada)
        if extrapolado is not None:
            ext = extrapolado.get(idx)
            if ext is not None and not ext.empty:
                ax.plot(ext.index, ext.values,
                        color=color, linewidth=2, linestyle="--", zorder=4,
                        label=f"{idx} proyectado")

        # Línea vertical: ventana de predicción
        if ventana_fecha is not None:
            etiq = f"{ventana_nombre} {ventana_fecha.strftime('%d/%m/%Y')}" if ventana_nombre else ventana_fecha.strftime('%d/%m/%Y')
            ax.axvline(x=ventana_fecha, color="#e74c3c", linewidth=1.5,
                       linestyle="--", alpha=0.8)
            ax.text(ventana_fecha, ax.get_ylim()[1], etiq,
                    color="#e74c3c", fontsize=9, ha="right", va="top")

        # Línea SOS
        if sos_fecha is not None:
            ax.axvline(x=sos_fecha, color="#2ecc71", linewidth=2,
                       linestyle="--", alpha=0.8)
            ax.text(sos_fecha, ax.get_ylim()[1], f"SOS {sos_fecha.strftime('%d/%m/%Y')}",
                    color="#2ecc71", fontsize=9, ha="left", va="top")

        # Línea EOS
        if eos_fecha is not None:
            ax.axvline(x=eos_fecha, color="#e74c3c", linewidth=2,
                       linestyle="--", alpha=0.8)
            ax.text(eos_fecha, ax.get_ylim()[1], f"EOS {eos_fecha.strftime('%d/%m/%Y')}",
                    color="#e74c3c", fontsize=9, ha="right", va="top")

        ax.set_ylabel(idx)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper right", fontsize=9)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(date_fmt)

    axes[-1].set_xlabel("Fecha")
    fig.suptitle(titulo, fontsize=14, y=1.01)
    plt.tight_layout()
    return fig, axes

---
### Ejemplo: cargar datos y generar gráfico

In [ ]:
# ── 4. Cargar datos de una parcela ────────────────────────────────────────────
# Misma lógica que Streamlit: cargar crudo + suavizado, extrapolación y validación

ID_PARCELA = 10
ID_CICLO = 2654

# Obtener fechas del ciclo
ciclo_row = pd.read_sql("""
    SELECT id_ciclo, id_parcela, temporada, sos, eos,
           fecha_inicio, fecha_fin, ventana, fecha_limite,
           ultima_prediccion_id
    FROM produccion_acumulada_ciclo
    WHERE id_ciclo = ?
""", get_connection_raw(), params=(ID_CICLO,))

if ciclo_row.empty:
    raise ValueError(f"Ciclo {ID_CICLO} no encontrado.")
c = ciclo_row.iloc[0]

SOS = pd.Timestamp(c["sos"])
EOS = pd.Timestamp(c["eos"])
VENTANA = c.get("ventana")
FECHA_LIMITE = pd.Timestamp(c["fecha_limite"]) if pd.notna(c.get("fecha_limite")) else None
ID_PREDICCION = c.get("ultima_prediccion_id")

print(f"Parcela {ID_PARCELA} — Ciclo #{ID_CICLO}: {c['temporada']}")
print(f"  SOS: {SOS.date()} → EOS: {EOS.date()}")
print(f"  Ventana: {VENTANA}, fecha_limite: {FECHA_LIMITE.date() if FECHA_LIMITE else '—'}")
print(f"  id_prediccion: {ID_PREDICCION}")

In [ ]:
# ── 5. Cargar índices (crudo + suavizado) ─────────────────────────────────────
dfs_crudos = cargar_indices_desde_bd(ids_parcelas=[ID_PARCELA])
dfs_suave = preprocesar_indices_vpm(dfs_crudos)
col = f"id_{ID_PARCELA}"

raw_evi = dfs_crudos["EVI"][col].dropna()
raw_lswi = dfs_crudos["LSWI"][col].dropna()

lo = min(raw_evi.index.min(), raw_lswi.index.min())
hi = max(raw_evi.index.max(), raw_lswi.index.max())
smooth_evi = dfs_suave["EVI"][col].loc[lo:hi]
smooth_lswi = dfs_suave["LSWI"][col].loc[lo:hi]

datos_ciclo = {
    "raw": {"EVI": raw_evi, "LSWI": raw_lswi},
    "smoothed": {"EVI": smooth_evi, "LSWI": smooth_lswi},
}
print(f"EVI crudo: {len(raw_evi)} pts, suavizado: {len(smooth_evi)} días")
print(f"LSWI crudo: {len(raw_lswi)} pts, suavizado: {len(smooth_lswi)} días")

In [ ]:
# ── 6. Cargar extrapolación y validación ───────────────────────────────────────
extrapolado = None
if ID_PREDICCION is not None:
    df_ext = pd.read_sql("""
        SELECT fecha, evi_extrapolado, lswi_extrapolado
        FROM series_extrapoladas_ventana
        WHERE id_prediccion = ?
        ORDER BY fecha
    """, get_connection_raw(), params=(int(ID_PREDICCION),), parse_dates=["fecha"])

    if not df_ext.empty:
        raw_ext = {}
        if df_ext["evi_extrapolado"].notna().any():
            raw_ext["EVI"] = df_ext.set_index("fecha")["evi_extrapolado"].dropna()
        if df_ext["lswi_extrapolado"].notna().any():
            raw_ext["LSWI"] = df_ext.set_index("fecha")["lswi_extrapolado"].dropna()
        if raw_ext:
            extrapolado = {}
            for idx in ("EVI", "LSWI"):
                smooth = datos_ciclo["smoothed"].get(idx)
                ext = raw_ext.get(idx)
                if ext is not None and smooth is not None and not smooth.empty:
                    last_smooth = smooth.iloc[-1]
                    extrapolado[idx] = pd.concat([
                        pd.Series([last_smooth], index=[smooth.index[-1]]),
                        ext,
                    ])
                elif ext is not None:
                    extrapolado[idx] = ext

validacion = {}
for idx in ("EVI", "LSWI"):
    serie = datos_ciclo["smoothed"].get(idx)
    if serie is not None and not serie.empty and FECHA_LIMITE is not None:
        tramo = serie.loc[FECHA_LIMITE:EOS]
        if not tramo.empty:
            validacion[idx] = tramo

print(f"Extrapolación: {'✓' if extrapolado else '✗'}")
print(f"Validación: {'✓' if any(not v.empty for v in validacion.values()) else '✗'}")

In [ ]:
# ── 7. Generar gráfico estático y exportar a PDF ──────────────────────────────
fig, axes = graficar_prediccion_estatica(
    datos=datos_ciclo,
    indices=["EVI", "LSWI"],
    extrapolado=extrapolado,
    validacion=validacion,
    ventana_fecha=FECHA_LIMITE,
    ventana_nombre=VENTANA,
    sos_fecha=SOS,
    eos_fecha=EOS,
    titulo=f"Parcela {ID_PARCELA} — Ciclo #{ID_CICLO}",
)

plt.savefig("prediccion_ventana.pdf", format="pdf", bbox_inches="tight")
plt.show()
print("PDF exportado: prediccion_ventana.pdf")

---
### Notas
- Cambia `ID_CICLO` en la celda 4 para probar otros ciclos.
- Cambia `set_db_path()` en la celda 2 para alternar entre BD real/pruebas.
- La función `graficar_prediccion_estatica` se puede reutilizar en otros notebooks o scripts.
- El PDF se guarda en el directorio de trabajo actual del notebook.